In [1]:
import numpy as np
import pandas as pd

import ta

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

In [2]:
HORIZON_DAYS = 90
DROP_THRESHOLD = 0.1

In [3]:
df = pd.read_csv('../data/test_crypto.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date').sort_index()
prices = df['BTC'].astype(float).dropna()
prices.name = 'Close'

print(f'Rows: {len(prices)} | Start: {prices.index.min().date()} | End: {prices.index.max().date()}')
prices.head()

Rows: 2243 | Start: 2020-02-13 | End: 2026-04-04


Date
2020-02-13    10214.379883
2020-02-14    10312.116211
2020-02-15     9889.424805
2020-02-16     9934.433594
2020-02-17     9690.142578
Name: Close, dtype: float64

# build X

In [4]:
def _build_features(prices: pd.Series) -> pd.DataFrame:
	ret = prices.pct_change()
	roll_max_30 = prices.rolling(30).max()
	roll_max_100 = prices.rolling(100).max()

	roll_min_30 = prices.rolling(30).min()
	roll_min_100 = prices.rolling(100).min()

	X = pd.DataFrame(index=prices.index)

	# Returns & volatility
	X["ret_30"] = prices.pct_change(30)
	X["ret_100"] = prices.pct_change(100)
     
	X["vol_30"] = ret.rolling(30).std()
	X["vol_100"] = ret.rolling(100).std()

	# Moving-average ratios
	X["ma_ratio_30"] = prices / prices.rolling(30).mean() - 1
	X["ma_ratio_100"] = prices / prices.rolling(100).mean() - 1

	# Drawdown and drawup
	X["drawdown_30"] = prices / roll_max_30 - 1
	X["drawdown_100"] = prices / roll_max_100 - 1

	X["drawup_30"] = prices / roll_min_30 - 1
	X["drawup_100"] = prices / roll_min_100 - 1

	# RSI (14)
	rsi = ta.momentum.RSIIndicator(close=prices, window=14)
	X["rsi_14"] = rsi.rsi() / 100

	# Stochastic RSI
	stoch_rsi = ta.momentum.StochRSIIndicator(close=prices, window=14)
	X["stoch_rsi_k"] = stoch_rsi.stochrsi_k()
	X["stoch_rsi_d"] = stoch_rsi.stochrsi_d()

	# MACD histogram (normalised by price)
	macd = ta.trend.MACD(close=prices)
	X["macd_diff"] = macd.macd_diff() / prices

	# Bollinger Bands %B and bandwidth
	bb = ta.volatility.BollingerBands(close=prices, window=20)
	X["bb_pct"] = bb.bollinger_pband()
	X["bb_width"] = bb.bollinger_wband()

	# Rate of Change (12-period)
	roc = ta.momentum.ROCIndicator(close=prices, window=12)
	X["roc_12"] = roc.roc() / 100
      
	return X

X = _build_features(prices)

# build y

In [5]:
future_return = prices.shift(-HORIZON_DAYS) / prices - 1
y = (future_return < -DROP_THRESHOLD).astype(int)

# dataset

In [6]:
dataset = pd.concat([X, y.rename('target')], axis=1)
dataset = dataset.dropna()
dataset['target'].value_counts(normalize=True)

target
0    0.70182
1    0.29818
Name: proportion, dtype: float64

In [7]:
X = dataset.drop(columns=['target'])
y = dataset['target']

In [8]:
X

,ret_30,ret_100,vol_30,vol_100,ma_ratio_30,ma_ratio_100,drawdown_30,drawdown_100,drawup_30,drawup_100,rsi_14,stoch_rsi_k,stoch_rsi_d,macd_diff,bb_pct,bb_width,roc_12
Date,,,,,,,,,,,,,,,,,
2020-05-23,0.239519,-0.098400,0.038344,0.056783,0.026368,0.157808,-0.074585,-0.106945,0.219628,0.852681,0.540063,0.067391,0.120376,-0.010895,0.421494,16.259180,0.070624
2020-05-24,0.164148,-0.147569,0.039517,0.056963,-0.024811,0.107260,-0.116681,-0.133270,0.161221,0.768405,0.472268,0.067391,0.077698,-0.014973,0.156183,16.594421,-0.001603
2020-05-25,0.176620,-0.099348,0.039535,0.056825,-0.016741,0.123333,-0.104967,-0.121777,0.159777,0.791856,0.491400,0.077510,0.070764,-0.015753,0.237699,16.796523,-0.039164
2020-05-26,0.150417,-0.110664,0.039586,0.056831,-0.028804,0.115814,-0.112190,-0.128865,0.133338,0.777395,0.479847,0.047120,0.064007,-0.016537,0.215704,17.432554,-0.092325
2020-05-27,0.177718,-0.052541,0.040030,0.056902,0.004129,0.160254,-0.077425,-0.094752,0.175989,0.846994,0.536345,0.160156,0.094929,-0.013371,0.449405,16.270885,-0.015778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-31,0.037957,-0.230061,0.026470,0.029630,-0.019680,-0.120933,-0.088534,-0.296051,0.034545,0.088214,0.472374,0.233541,0.091837,-0.005192,0.331408,13.960472,-0.024022
2026-04-01,-0.010139,-0.230664,0.025095,0.029630,-0.021577,-0.120615,-0.090602,-0.297648,0.032198,0.085746,0.468207,0.413896,0.229802,-0.003672,0.329749,14.134749,-0.034656
2026-04-02,-0.020574,-0.234807,0.025268,0.029653,-0.038032,-0.133689,-0.106497,-0.309924,0.014156,0.066768,0.436333,0.466491,0.371309,-0.003628,0.236111,14.517982,-0.026531


In [9]:
y

Date
2020-05-23    0
2020-05-24    0
2020-05-25    0
2020-05-26    0
2020-05-27    0
             ..
2026-03-31    0
2026-04-01    0
2026-04-02    0
2026-04-03    0
2026-04-04    0
Name: target, Length: 2143, dtype: int64

In [10]:
model = RandomForestClassifier(
    bootstrap=True,
    oob_score=True,
    class_weight='balanced_subsample',
    n_jobs=-1,
)
model.fit(X, y)

RandomForestClassifier(class_weight='balanced_subsample', n_jobs=-1,
                       oob_score=True)

In [11]:
oob_prob = model.oob_decision_function_[:, 1]
oob_pred = (oob_prob >= 0.5).astype(int)

print('Total rows:', len(X))
print('Positive rate:', round(y.mean(), 3))
print('ROC AUC (OOB):', round(roc_auc_score(y, oob_prob), 4))
print('\nClassification report (OOB)\n')
print(classification_report(y, oob_pred, zero_division=0))

Total rows: 2143
Positive rate: 0.298
ROC AUC (OOB): 0.9894

Classification report (OOB)

              precision    recall  f1-score   support

           0       0.97      0.98      0.97      1504
           1       0.95      0.93      0.94       639

    accuracy                           0.96      2143
   macro avg       0.96      0.95      0.96      2143
weighted avg       0.96      0.96      0.96      2143



In [12]:
print(confusion_matrix(y, oob_pred))

[[1470   34]
 [  46  593]]
